# LaDe joint model on Colab GPU — no Drive mount
Use this when Drive mounting / `auth.authenticate_user` fails.
1. **Runtime > Change runtime type > T4 GPU**.
2. Get the zip into the session with **one** of the options in cell 2:
   - **A (recommended):** in Google Drive, right-click `lade_colab.zip` > **Share** > General access: *Anyone with the link* > **Copy link**, paste it below.
   - **B:** drag `lade_colab.zip` from your PC into the **Files** panel (folder icon, left) and wait until the upload finishes (537 MB, can be slow).
3. **Nothing is saved automatically** (no Drive). Run the *download results* cell after each finished run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

In [ ]:
import os
DRIVE_LINK = ''   # option A: paste the share link here; leave empty for option B (uploaded to /content)
os.chdir('/content')
if not os.path.exists('/content/lade_colab.zip'):
    assert DRIVE_LINK, 'paste the share link into DRIVE_LINK, or upload lade_colab.zip into /content'
    !gdown --fuzzy "$DRIVE_LINK" -O /content/lade_colab.zip
!ls -la /content/lade_colab.zip

In [ ]:
!cd /content && unzip -qo lade_colab.zip
# newer code: upload lade_code_<version>.zip (small) to /content; the newest name wins and overwrites the code in the data zip
import glob, os
code_zips = sorted(glob.glob('/content/lade_code*.zip'))
if code_zips:
    print('applying', os.path.basename(code_zips[-1]))
    !cd /content && unzip -qo "{code_zips[-1]}"
else:
    print('WARNING: no lade_code_*.zip uploaded -> running the OLD code inside lade_colab.zip')
%cd /content/lade-route-eta
!echo "CODE VERSION:" && cat CODE_VERSION.txt 2>/dev/null || echo "none (old code)"
!mkdir -p results/models
!pip -q install -r colab/requirements_colab.txt
import torch, pandas; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| pandas', pandas.__version__)

## Check: smoke test (tiny data, 1 epoch, CPU), should print metrics

In [ ]:
!python scripts/train_joint.py --dataset pickup --smoke

## Real run (GPU): one seed first
Check time per epoch and RAM (top right). If it crashes while building data, use `--frac 0.1`.
Settings (decided on validation, 2026-11-16): pickup `--select krc`, delivery `--select mae` (the scored task chooses the saved model), `--patience 5`. The log is saved to `results/` too.

In [ ]:
!python -u scripts/train_joint.py --dataset pickup --frac 0.25 --seed 0 --select krc --patience 5 2>&1 | tee results/log_pickup_s0.txt

## Download results (run after every finished run — the session disk is wiped on disconnect)

In [ ]:
!cd /content/lade-route-eta && zip -qr /content/lade_results.zip results
from google.colab import files
files.download('/content/lade_results.zip')

## All 5 seeds, both datasets
Downloads the results zip after each run, so a disconnect loses at most the current run.
(Allow multiple downloads if the browser asks.)

In [ ]:
import os
from google.colab import files
for ds in ['pickup', 'delivery']:
    sel = 'krc' if ds == 'pickup' else 'mae'   # pickup scored on route, delivery on ETA
    for s in range(5):
        if os.path.exists(f'results/joint_{ds}_h64_l3_lam1.0_s{s}.json'):
            print('done, skipping', ds, s); continue
        !python -u scripts/train_joint.py --dataset {ds} --frac 0.25 --seed {s} --select {sel} --patience 5 2>&1 | tee results/log_{ds}_s{s}.txt
        !cd /content/lade-route-eta && zip -qr /content/lade_results.zip results
        files.download('/content/lade_results.zip')